# Частина 2: Аналіз датасету Individual Household Electric Power Consumption
Звантаження, зчитування та очищення даних (Data Cleaning).

In [3]:
import pandas as pd
import numpy as np
import timeit
import urllib.request
import zipfile
import os

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
zip_path = "household_power_consumption.zip"
txt_path = "household_power_consumption.txt"

if not os.path.exists(txt_path):
    urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall()

df = pd.read_csv(txt_path, sep=';', na_values=['?'], low_memory=False)

df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], format='%d/%m/%Y %H:%M:%S')
df.set_index('Datetime', inplace=True)
df.drop(['Date', 'Time'], axis=1, inplace=True)

df.dropna(inplace=True)

print(f"Дані успішно очищені. Залишилось записів: {len(df)}")
df.head()

Дані успішно очищені. Залишилось записів: 2049280


,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
Datetime,,,,,,,
2006-12-16 17:24:00,4.216,0.418,234.84,18.4,0.0,1.0,17.0
2006-12-16 17:25:00,5.360,0.436,233.63,23.0,0.0,1.0,16.0
2006-12-16 17:26:00,5.374,0.498,233.29,23.0,0.0,2.0,17.0
2006-12-16 17:27:00,5.388,0.502,233.74,23.0,0.0,1.0,17.0
2006-12-16 17:28:00,3.666,0.528,235.68,15.8,0.0,1.0,17.0


## Формування вибірок
Функції для виконання чотирьох завдань з фільтрації та обчислень.

In [5]:
def task_1_high_power(data):
    return data[data['Global_active_power'] > 5.0]

def task_2_current_and_appliances(data):
    return data[(data['Global_intensity'] >= 19.0) & 
                (data['Global_intensity'] <= 20.0) & 
                (data['Sub_metering_2'] > data['Sub_metering_3'])]

def task_3_random_sample_mean(data):
    sample = data.sample(n=500000, replace=False, random_state=42)
    return sample[['Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']].mean()

def task_4_complex_filter(data):
    res = data[(data.index.hour >= 18) & (data['Global_active_power'] > 6.0)]
    
    res = res[(res['Sub_metering_2'] > res['Sub_metering_1']) & 
              (res['Sub_metering_2'] > res['Sub_metering_3'])]
    
    half_idx = len(res) // 2
    first_half = res.iloc[:half_idx]
    second_half = res.iloc[half_idx:]
    
    res1 = first_half.iloc[::3]
    res2 = second_half.iloc[::4]
    
    return pd.concat([res1, res2])

## Виконання завдань та профілювання часу (timeit)
Аналіз часових витрат на виконання процедур.

In [6]:
def profile_task(task_func, data, task_name):
    start_time = timeit.default_timer()
    result = task_func(data)
    execution_time = timeit.default_timer() - start_time
    
    print(f"{task_name}")
    print(f"Час виконання: {execution_time:.4f} секунд")
    if isinstance(result, pd.Series):
        print("Результат (Середні значення):\n", result, "\n")
    else:
        print(f"Знайдено записів: {len(result)}\n")
    return result

res1 = profile_task(task_1_high_power, df, "Завдання 1: Потужність > 5 кВт")
res2 = profile_task(task_2_current_and_appliances, df, "Завдання 2: Струм 19-20 А + Група 2 > Групи 3")
res3 = profile_task(task_3_random_sample_mean, df, "Завдання 3: Середнє для 500,000 випадкових записів")
res4 = profile_task(task_4_complex_filter, df, "Завдання 4: Складний вечірній фільтр")

Завдання 1: Потужність > 5 кВт
Час виконання: 0.0056 секунд
Знайдено записів: 17547

Завдання 2: Струм 19-20 А + Група 2 > Групи 3
Час виконання: 0.0092 секунд
Знайдено записів: 2509

Завдання 3: Середнє для 500,000 випадкових записів
Час виконання: 0.0635 секунд
Результат (Середні значення):
 Sub_metering_1    1.119258
Sub_metering_2    1.308912
Sub_metering_3    6.452950
dtype: float64 

Завдання 4: Складний вечірній фільтр
Час виконання: 0.0290 секунд
Знайдено записів: 310



## Статистичні операції
* Нормування та стандартизація
* Кореляції Пірсона та Спірмена
* One Hot Encoding категоріального атрибута

In [8]:
numeric_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity']

df_normalized = (df[numeric_cols] - df[numeric_cols].min()) / (df[numeric_cols].max() - df[numeric_cols].min())

df_standardized = (df[numeric_cols] - df[numeric_cols].mean()) / df[numeric_cols].std()

print("Перші рядки стандартизованих даних:")
print(df_standardized.head(), "\n")

pearson_corr = df['Global_active_power'].corr(df['Global_intensity'], method='pearson')
spearman_corr = df['Global_active_power'].corr(df['Global_intensity'], method='spearman')

print(f"Кореляція Пірсона (Потужність vs Струм): {pearson_corr:.4f}")
print(f"Кореляція Спірмена (Потужність vs Струм): {spearman_corr:.4f}\n")

df_encoded = df.copy()
df_encoded['Day_of_week'] = df_encoded.index.day_name()

df_ohe = pd.get_dummies(df_encoded, columns=['Day_of_week'], prefix='Day')

print("Стовпці після One Hot Encoding:")
print(df_ohe.filter(like='Day_').head())

Перші рядки стандартизованих даних:
                     Global_active_power  Global_reactive_power   Voltage  \
Datetime                                                                    
2006-12-16 17:24:00             2.955076               2.610720 -1.851816   
2006-12-16 17:25:00             4.037084               2.770405 -2.225274   
2006-12-16 17:26:00             4.050325               3.320431 -2.330213   
2006-12-16 17:27:00             4.063566               3.355916 -2.191323   
2006-12-16 17:28:00             2.434881               3.586572 -1.592555   

                     Global_intensity  
Datetime                               
2006-12-16 17:24:00          3.098788  
2006-12-16 17:25:00          4.133799  
2006-12-16 17:26:00          4.133799  
2006-12-16 17:27:00          4.133799  
2006-12-16 17:28:00          2.513781   



ModuleNotFoundError: No module named 'scipy'